In [0]:
from pyspark.sql import functions as f,Window
import re
from datetime import datetime
from pyspark.sql.types import *

c_schema = StructType([
    StructField('customer_id',IntegerType()),
    StructField('customer_name',StringType()),
    StructField('city',StringType()),
    StructField('customer_segment',StringType()),
    StructField('email',StringType()),
    StructField('phone',StringType())
])

source_path = '/Volumes/madstagescd2/madstage/madstage/Initialfile/'
files = dbutils.fs.ls(source_path)
customer_files = []

for file in files:
    match = re.search(r'(\d{4}-\d{2}-\d{2})',file.name)
    if file.name.startswith('customer'):
        if match:
            file_time = datetime.strptime(match.group(1),"%Y-%m-%d")
            customer_files.append((file_time,file.path))

if customer_files:
    latest_file = max(customer_files,key=lambda x:x[0])
    print(latest_file[0],latest_file[1])
    customer_df = (spark.read.format('csv').option('header','true').schema(c_schema).load(latest_file[1]))

customer_df = customer_df.dropna(subset=('customer_id'))
customer_df = customer_df.dropDuplicates()
# customer_id = customer_id.withColumn('rn',f.row_number().over(Window.partitionBy('customer_id').orderBy()))
customer_df = customer_df.fillna({
    'customer_name':'Unknown',
    'city':'Unknown',
    'customer_segment':'Unknown',
    'email':'Unknown',
    'phone':0
})

phone_regex = r'[^0-9]'
customer_df = customer_df.withColumn('customer_name',f.initcap(f.trim(f.col('customer_name')))).withColumn('city',f.initcap(f.trim(f.col('city')))).withColumn('customer_segment',f.initcap(f.trim(f.col('customer_segment')))).withColumn('email',f.lower(f.trim(f.col('email')))).withColumn('phone',f.regexp_replace(f.col('phone'),phone_regex,''))

email_regex = r'^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$'
email_validation =(f.col('email').rlike(email_regex))
phone_validation = (f.length(f.col('phone'))==10)

customer_df = customer_df.withColumn('dataValidation',f.when((email_validation) & (phone_validation),'Pass').otherwise('Fail'))

customer_df = customer_df.withColumn('is_current',f.lit('Y')).withColumn('updated_dt',f.lit('').cast('string')).withColumn('created_dt',f.current_date().cast('string'))

silver = customer_df.filter(f.col('dataValidation')=='Pass').drop('dataValidation')
silver.show()

silver.write.format('delta').mode('overwrite').option('overwriteSchema','true').saveAsTable('madstagescd2.madstage.silver_customers')



